# EgoHandBench demo

In [1]:
from pathlib import Path

import numpy as np
from IPython.display import Video, display
from tqdm.auto import tqdm

from baselines.hand_detector import HandDetector
from utils.io import load_npz, load_sample, save_npz, save_video
from utils.visualizer import Visualizer

visualizer = Visualizer()

## ViDiHand2

In [2]:
model = HandDetector("vidihand_v2")
output = Path("results/sample/vidihand_v2")

### Ego4D

In [3]:
sample = load_sample("datasets/sample/ego4d.mp4", "datasets/sample/ego4d_camera.json")
prediction = model.inference_offline(sample)
preview = np.concatenate((visualizer.render_mano(prediction["mano"], sample), visualizer.render_joint(prediction["joint"], sample)), axis=2)
save_video(output / "ego4d.mp4", preview, sample["camera"].fps)
display(Video(filename=str(output / "ego4d.mp4"), embed=True))

ViDiHand2: VAE encoding:   0%|          | 0/301 [00:00<?, ?frame/s]

ViDiHand2: VAE encoding:   0%|          | 1/301 [00:00<01:21,  3.70frame/s]

ViDiHand2: VAE encoding:   7%|▋         | 21/301 [00:00<00:04, 67.04frame/s]

ViDiHand2: VAE encoding:  11%|█         | 33/301 [00:00<00:03, 78.70frame/s]

ViDiHand2: VAE encoding:  15%|█▍        | 45/301 [00:00<00:02, 85.61frame/s]

ViDiHand2: VAE encoding:  19%|█▉        | 57/301 [00:00<00:02, 90.05frame/s]

... (51 outputs left out)

ViDiHand2: fitting MANO to the predicted joints:   0%|          | 0/60 [00:00<?, ?step/s]

ViDiHand2: fitting MANO to the predicted joints:   2%|▏         | 1/60 [00:00<00:09,  6.13step/s]

ViDiHand2: fitting MANO to the predicted joints: 100%|██████████| 60/60 [00:00<00:00, 253.46step/s]

ViDiHand2: fitting MANO to the predicted joints: 100%|██████████| 60/60 [00:00<00:00, 210.53step/s]

### Gen-HumanEgo

In [4]:
sample = load_sample("datasets/sample/gen_humanego.mp4", "datasets/sample/gen_humanego_camera.json")
prediction = model.inference_offline(sample)
preview = np.concatenate((visualizer.render_mano(prediction["mano"], sample), visualizer.render_joint(prediction["joint"], sample)), axis=2)
save_video(output / "gen_humanego.mp4", preview, sample["camera"].fps)
display(Video(filename=str(output / "gen_humanego.mp4"), embed=True))

ViDiHand2: VAE encoding:   0%|          | 0/301 [00:00<?, ?frame/s]

ViDiHand2: VAE encoding:   4%|▍         | 13/301 [00:00<00:02, 119.22frame/s]

ViDiHand2: VAE encoding:   8%|▊         | 25/301 [00:00<00:02, 110.25frame/s]

ViDiHand2: VAE encoding:  12%|█▏        | 37/301 [00:00<00:02, 109.86frame/s]

ViDiHand2: VAE encoding:  16%|█▋        | 49/301 [00:00<00:02, 109.75frame/s]

ViDiHand2: VAE encoding:  20%|██        | 61/301 [00:00<00:02, 109.62frame/s]

... (27 outputs left out)

ViDiHand2: fitting MANO to the predicted joints:   2%|▏         | 1/60 [00:00<00:09,  6.47step/s]

ViDiHand2: fitting MANO to the predicted joints:  35%|███▌      | 21/60 [00:00<00:00, 92.31step/s]

ViDiHand2: fitting MANO to the predicted joints: 100%|██████████| 60/60 [00:00<00:00, 192.49step/s]

ViDiHand2: fitting MANO to the predicted joints: 100%|██████████| 60/60 [00:00<00:00, 153.00step/s]

## WiLoR

In [5]:
del model
model = HandDetector("wilor", version="official")  # e.g. HandDetector("hamer", version="egohandbench"), HandDetector("ace_ego_hand")
output = Path("results/sample/wilor/official")

### H2O

In [6]:
sample = load_sample("datasets/sample/h2o.mp4", "datasets/sample/h2o_camera.json")
prediction = model.inference_offline(sample)

for source, arrays in prediction.items():
    save_npz(output / source / (sample["name"] + ".npz"), arrays)
prediction = {source: load_npz(output / source / (sample["name"] + ".npz")) for source in ("mano", "joint")}

mesh = visualizer.render_mano(prediction["mano"], sample)
joints = visualizer.render_joint(prediction["joint"], sample)
preview = np.concatenate((mesh, joints), axis=2)
save_video(output / "video.mp4", preview, sample["camera"].fps)
display(Video(filename=str(output / "video.mp4"), embed=True))


Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at /pytorch/aten/src/ATen/native/Cross.cpp:62.)


### H2O, streaming

In [7]:
video, camera = sample["video"], sample["camera"]
preview = []
for i in tqdm(range(len(video)), desc="Streaming", unit="frame"):
    frame = {"image": video[i], "camera": camera.slice([i])}
    prediction = model.inference_stream(frame)
    mesh = visualizer.render_mano(prediction["mano"], frame)
    joints = visualizer.render_joint(prediction["joint"], frame)
    preview.append(np.concatenate((mesh[0], joints[0]), axis=1))
save_video(output / "stream.mp4", np.asarray(preview), camera.fps)
display(Video(filename=str(output / "stream.mp4"), embed=True))

Streaming:   0%|          | 0/81 [00:00<?, ?frame/s]

Streaming:   1%|          | 1/81 [00:00<00:37,  2.13frame/s]

Streaming:   2%|▏         | 2/81 [00:00<00:38,  2.04frame/s]

Streaming:   4%|▎         | 3/81 [00:01<00:39,  2.00frame/s]

Streaming:   5%|▍         | 4/81 [00:01<00:37,  2.04frame/s]

Streaming:   6%|▌         | 5/81 [00:02<00:37,  2.05frame/s]

... (73 outputs left out)

Streaming:  98%|█████████▊| 79/81 [00:37<00:00,  2.05frame/s]

Streaming:  99%|█████████▉| 80/81 [00:38<00:00,  2.05frame/s]

Streaming: 100%|██████████| 81/81 [00:38<00:00,  2.04frame/s]

Streaming: 100%|██████████| 81/81 [00:38<00:00,  2.10frame/s]